In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### CMIP6 data from Copernicus CDS

The **Copernicus Climate Data Store (CDS)** provides CMIP6 climate projections via a simple API.
It is the easiest entry point for CMIP6 data — you request a region and receive pre-subsetted NetCDF files.

#### 🌡️ CMIP6 emission scenarios (SSPs)

| Scenario | Radiative forcing 2100 | ΔT by 2100 (likely range) | Narrative |
|----------|----------------------|--------------------------|----------|
| **SSP1-2.6** | 2.6 W/m² | +1.0 – +1.8 °C | Sustainability, strong mitigation |
| **SSP2-4.5** | 4.5 W/m² | +2.1 – +3.5 °C | Middle of the road |
| **SSP3-7.0** | 7.0 W/m² | +2.8 – +4.6 °C | Regional rivalry, delayed action |
| **SSP5-8.5** | 8.5 W/m² | +3.3 – +5.7 °C | Fossil-fuelled growth |

> For infrastructure design: **SSP2-4.5** as central estimate + **SSP5-8.5** for worst-case resilience.
> The number in the name (2.6, 4.5…) is the radiative forcing in W/m² reached by 2100.

#### 🔑 Setting up CDS credentials

1. Register at https://cds.climate.copernicus.eu (free)
2. Go to your profile → copy your **Personal Access Token**
3. Create `~/.cdsapirc`:
   ```ini
   url: https://cds.climate.copernicus.eu/api
   key: <your-personal-access-token>
   ```
4. `pip install cdsapi`

> Downloads are **queued** on the CDS server. Simple requests (1 model, small region) complete in minutes;
> large ensemble downloads (all models, long period) can take hours.

#### 💡 Key characteristics

- **Provider:** Copernicus Climate Change Service (C3S), operated by ECMWF
- **Scenarios:** Historical + SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
- **Spatial resolution:** ~1° or ~0.5° (regridded from native model resolution)
- **Temporal resolution:** Monthly or daily
- **Coverage:** Historical 1850–2014; projections 2015–2100

#### 🔗 Access
- Portal: https://cds.climate.copernicus.eu/
- Dataset: https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-cmip6


In [2]:
from pyhydra.utils import interactive_map
from pyhydra.data_sources.climate_change import download_CDS_CMIP6
import os
from pathlib import Path

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))

RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
OUTPUT_DIR = Path(os.getenv('HYDRA_RUNTIME_DIR', str(Path.cwd() / '.hydra_runtime'))) / 'cmip6'
print(f'Run downloads: {RUN_DOWNLOADS} | Output dir: {OUTPUT_DIR}')


Run downloads: False | Output dir: /Users/salvadornavasfernandez/Desktop/Github/HYDRA/notebooks/data_sources/climate_change/.hydra_runtime/cmip6


## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the bounding box. Read `coordinates_list[0]` in the configuration cell below.

In [3]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

---
## ⚙️ Download parameters

```python
download_CDS_CMIP6(
    url='https://cds.climate.copernicus.eu/api',
    api_key='<your-key>',          # from ~/.cdsapirc
    start_date='1950-01-01',       # historical starts 1850; use 1950+ for practical record lengths
    end_date='2014-12-31',         # historical ends 2014; SSP starts 2015
    temporal_resolution='monthly', # 'monthly' (smaller files) or 'daily' (needed for extremes)
    model='All',                   # or e.g. 'MIROC6' to limit to one model
    experiments=['historical', 'ssp2_4_5', 'ssp5_8_5'],
    variables=['precipitation'],
    download_base_dir='cmip6_data/',
    area=[44, -10, 35, 5],         # [N, W, S, E] — Iberian Peninsula
)
```

#### Choosing `temporal_resolution`

| Use case | Resolution | Reason |
|----------|------------|--------|
| Trend analysis, climatology | Monthly | Much smaller files (~12× less data) |
| Bias correction calibration | Monthly | QM/QDM only needs CDFs, not daily timing |
| Extreme value analysis | **Daily** | Annual maxima require daily resolution |
| Hydrological model forcing | **Daily** | Event timing and intensity needed |

> ⚠️ Daily global CMIP6 for all models can be **several TB**. Always subset to your region first.


In [4]:
# === GENERAL PARAMETERS ===

url     = 'https://cds.climate.copernicus.eu/api'
api_key = 'YOUR_CDS_API_KEY'   # https://cds.climate.copernicus.eu/profile

temporal_resolution = 'monthly'
model               = 'All'
experiments         = ['historical', 'ssp2_4_5', 'ssp5_8_5']
variables = [
    'precipitation',
    'daily_maximum_near_surface_air_temperature',
    'daily_minimum_near_surface_air_temperature',
]

# Output base directory — portable across local clone, Docker and Azure ML
download_base_dir = str(OUTPUT_DIR)

area = coordinates_list[0] if coordinates_list else [39.84, -12.04, 33.50, 2.29]

model_skip_list = set()

if not RUN_DOWNLOADS:
    print('[CDS] Download skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.')
elif api_key == 'YOUR_CDS_API_KEY':
    print('[CDS] API key not configured — set api_key above to download.')
    print('[CDS] Register at: https://cds.climate.copernicus.eu/profile')
else:
    for variable in variables:
        for experiment in experiments:
            if experiment == 'historical':
                start_date, end_date = '1950-01-01', '2014-12-31'
            else:
                start_date, end_date = '2015-01-01', '2099-12-31'

            var_map = {
                'precipitation':                                  'pr',
                'daily_maximum_near_surface_air_temperature':     'tasmax',
                'daily_minimum_near_surface_air_temperature':     'tasmin',
            }
            download_dir = os.path.join(download_base_dir, var_map.get(variable, variable), experiment)
            os.makedirs(download_dir, exist_ok=True)

            if model in model_skip_list:
                print(f'⚠️ Skipping model {model} due to previous critical errors.')
                continue

            try:
                download_CDS_CMIP6(
                    url=url,
                    api_key=api_key,
                    start_date=start_date,
                    end_date=end_date,
                    temporal_resolution=temporal_resolution,
                    variables=[variable],
                    model=model,
                    experiments=[experiment],
                    area=area,
                    download_base_dir=download_dir,
                    max_workers=5,
                )
            except RuntimeError as e:
                if 'RoocsValueError' in str(e):
                    print(f'❌ Critical error for model {model}: {e}. Skipping.')
                    model_skip_list.add(model)


[CDS] Download skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 to run it.


---
## 📊 Interpreting the climate change signal

After downloading historical + SSP data, the standard analysis workflow is:

**Temperature** (robust signal — models agree on direction):
```python
# Change in mean annual temperature
delta_T = ds_ssp.tasmax.sel(time=slice('2070','2100')).mean() \
        - ds_hist.tasmax.sel(time=slice('1981','2010')).mean()
```

**Precipitation** (uncertain — models disagree in sign for many regions):
```python
# Relative change (%)
delta_P = (ds_ssp.pr.mean() - ds_hist.pr.mean()) / ds_hist.pr.mean() * 100
```
Only report precipitation changes as robust when **≥80% of models agree on the sign**.

#### Before using in hydrological modelling

1. ✅ Validate historical simulation against ERA5 or AEMET
2. 🔧 Apply bias correction (QDM) — see `bias_correction` notebook
3. 📈 Propagate corrected change signal through rainfall-runoff model
4. 📊 Report uncertainty range (10th–90th percentile across models)
